# Hierarchical Clustering - UPGMA

In [ ]:
from typing import List, Tuple, Dict
from io import StringIO

import matplotlib.pyplot as plt
from Bio import Phylo
import numpy as np

In [ ]:
def read_fasta(filename: str) -> Dict[str, str]:
    """Reads sequences from FASTA file.

    Args:
        filename (str): Path to FASTA file containing HIV RT sequences

    Returns:
        Dict[str, str]: Dictionary mapping sequence IDs to sequences

    Examples:
        >>> seqs = read_fasta("lafayette_SARS_RT.fasta")
        >>> len(seqs) > 0
        True
    """
    #sequences = empty dictionary (name to sequence)
    #current_name = nothing yet

    #Open the file and read it line by line:
        #strip spaces and the newline off the end of the line
        #IF the line is empty:
            #skip it
        #IF the line starts with ">":
            #current_name = last word of the line (i.e. "DM1")
            #sequences[current_name] = empty string
        #ELSE (it's part of a sequence):
            #add the line to the end of sequences[current_name]

    #Return sequences

In [ ]:
# Changes from Project 05 smith_waterman:
    # 1. cal_score is kept as-is; it still calculates each cell's score. The traceback arrow it records is not used here.
    # 2. Drop traceback, along with the traceback_matrix lines and the max_pos line, since nothing uses them anymore.
    # 3. Keep the original table-filling loop and the best_score line in a helper function, and remove both print statements.
    # 4. smith_waterman uses the Project 06 name and inputs, calls the helper, and returns the normalized score:
    #    normalized = raw(seq1, seq2) / smaller of [raw(seq1, seq1), raw(seq2, seq2)]

def cal_score(matrix, seq1, seq2, i, j, match, mismatch, gap):  #Original calc_score from Project05 untouched.  Needed to run Smith_Waterman function from Project05
    '''Calculate score for position (i,j) in scoring matrix, also record move to trace back
    
    Args:
        matrix (numpy array): scoring matrix
        seq1 (str): sequence 1
        seq2 (str): sequence 2
        i (int): current row number
        j (int): current column number
        
    Returns:
        score in position (i,j)    
        move to trace back: 0-END, 1-DIAG, 2-UP, 3-LEFT
        
    Pseudocode:
        Calculate scores based on upper-left, up, and left neighbors:
            diag_score = upper-left + (match or mismatch)
            up_score = up + gap
            left_score = left + gap
        score = max(0, diag_score, up_score, left_score)
        traceback = maximum direction or end
        
    '''
    row_letter = seq1[i-1]  #Get this row's letter: seq1 at position (i-1)
    col_letter = seq2[j-1]  #Get this column's letter: seq2 at position (j-1)
    
    #Diagonal number:
    if row_letter == col_letter:    #If this row's letter equals this column's letter
        diag_score = matrix[i - 1, j - 1] + match  #diag_score = matrix at (i-1, j-1) + match
    else:        #Otherwise:
        diag_score = matrix[i - 1, j - 1] + mismatch  #diag_score = matrix at (i-1, j-1) + mismatch
    up_score = matrix[i - 1, j] + gap  #up_score = matrix at (i-1, j) + gap
    left_score = matrix[i, j-1] + gap  #left_score = matrix at (i, j-1) + gap

    score = max(0, diag_score, up_score, left_score)  #score = biggest of: 0, diag_score, up_score, left_score

    #Pick the move (check in this order, first match wins if tie score):
    if score == 0:   #IF score is 0:  move = 0 (END)
        move = 0
    elif score == diag_score:  #ELIF score = diag:  move = 1 (DIAG)
        move = 1
    elif score == up_score:  #ElIF score = up:  move = 2 (UP)
        move = 2
    else:    #ELSE:  move = 3 (LEFT)
        move = 3

    return score, move  #Return score and move


def project5_smith_waterman_score(seq1, seq2, match, mismatch, gap):  #Original smith_waterman function from Project05, turned into a helper function for the new smith_waterman function in Project06
    '''Fill the Smith-Waterman score table and return only the highest score
    
    Args:
        seq1 (str): first sequence
        seq2 (str): second sequence
        match (float): score for matching letters
        mismatch (float): penalty for mismatched letters
        gap (float): penalty for a gap
        
    Returns:
        float: highest score in the table (the raw alignment score)
    '''
    rows = len(seq1) + 1  #rows = length of seq1 + 1
    cols = len(seq2) + 1  #cols = length of seq2 + 1
    score_matrix = np.zeros((rows, cols))  #score_matrix = grid of zeros, size rows x cols (decimals)
    
    for i in range(1, rows):    #for i from 1 to rows -1:
        for j in range(1, cols):    #for j from 1 to cols -1:
            score, move = cal_score(score_matrix, seq1, seq2, i, j, match, mismatch, gap)  #score, move = cal_score(...); move is not used here
            score_matrix[i, j] = score   #score_matrix at (i, j) = score
    
    best_score = np.max(score_matrix)    #best_score = highest number in score matrix
    
    return float(best_score)   #Return best_score as a plain decimal number


def smith_waterman(
    sequence1: str,
    sequence2: str,
    match_score: float = 2.0,
    mismatch_penalty: float = -1.0,
    gap_penalty: float = -1.0
) -> float:
    """Implements Smith-Waterman local alignment algorithm.

    Args:
        sequence1 (str): First HIV RT sequence
        sequence2 (str): Second HIV RT sequence
        match_score (float): Score for matching characters
        mismatch_penalty (float): Penalty for mismatched characters
        gap_penalty (float): Penalty for gaps

    Returns:
        float: Normalized similarity score between sequences

    Examples:
        >>> seq1 = "ACGT"
        >>> seq2 = "ACGT"
        >>> smith_waterman(seq1, seq2)
        1.0
    """
    #raw_score = project5_smith_waterman_score(sequence1 vs sequence2, using match_score, mismatch_penalty, gap_penalty)
    #self_score1 = project5_smith_waterman_score(sequence1 vs sequence1, same scores)
    #self_score2 = project5_smith_waterman_score(sequence2 vs sequence2, same scores)

    #smaller_self_score = smaller of self_score1 and self_score2

    #IF smaller_self_score is 0 (only happens if a sequence is empty):
        #return 0.0 (avoids dividing by zero)
    
    #Return (raw_score / smaller_self_score) #Note: 1.0 = identical, closer to 0 = less similar


In [ ]:
def build_distance_matrix(sequences: Dict[str, str]) -> Tuple[np.ndarray, List[str]]:
    """Builds DISTANCE (not similiarity) matrix using Smith-Waterman scores.

    Args:
        sequences (Dict[str, str]): Dictionary of HIV RT sequences

    Returns:
        Tuple[np.ndarray, List[str]]: Distance matrix and list of sequence IDs

    Examples:
        >>> seqs = read_fasta("lafayette_SARS_RT.fasta")
        >>> mat, ids = build_distance_matrix(seqs)
        >>> mat[0:3, 0:3]  # Show 3x3 slice of distance matrix
        array([[0.000, 0.004, 0.012],
              [0.004, 0.000, 0.012],
              [0.012, 0.012, 0.000]])
        >>> len(ids)  # Number of sequences
        19
    """
    #ids = list of the sequence names, in file order (i.e. ["DM1", "DM2", ...])
    #n = number of sequences
    #distance_matrix = n x n grid of zeros (diagonal stays 0: each sequence vs itself)

    #For each row i from 0 to n-1:
        #FOR each column j from i+1 to n-1 (only pairs not yet done; skip the diagonal):
            #similarity = smith_waterman (sequence for ids[i], sequence for ids[j])
            #distance = 1 - similarity ***my suggestion for the group because can't produce undefined values and for those highly similar sequences it gives nearly the same distances as -log(similarity) - add to README
            #distance_matrix[i, j] = distance
            #distance_matrix[j, i] = distance  (Table is symmetric, so fill both halves)

    #Return distance_matrix, ids

In [ ]:
def neighbor_joining(
    distance_matrix: np.ndarray,
    labels: List[str]
) -> str:
    """Implements Neighbor-Joining algorithm for phylogenetic tree construction.

    Args:
        distance_matrix (np.ndarray): Distance matrix from Smith-Waterman scores
        labels (List[str]): Sequence identifiers

    Returns:
        str: Newick format tree string

    Examples:
        >>> seqs = read_fasta("lafayette_SARS_RT.fasta")
        >>> mat, ids = build_distance_matrix(seqs)
        >>> tree = neighbor_joining(mat, ids)
        >>> tree.startswith("((DM1:0.002,DM2:0.002)")  # Start of Newick string
        True
        >>> tree.count(",")  # Number of separators in tree
        18
    """

    #MY PROPOSAL FOR DECISIONS IN PSEUDOCODE/CODE
        #Final branch between the last two nodes is split evenly between them
        #Branch lengths are written to 4 decimals places in the Newick text (calculations use full precision)
        #All 20 sequences are used.  The docstring expects 19, note it in README
        #Ties in the D* score: the first pair found is kept
        #Negative branch lengths: set to 0, differences added to the other branch


    #SETUP
    #D = copy of distance_matrix (so original table isn't changed)
    #current_labels = copy of labels (each starts as just a name, i.e. "DM1")

    #WHILE more than 2 nodes remain:
        #n = number of rows in D

        #Step 1. Total Distance:  totals = sum of each for row of D

        #Step 2. D* score for every pair, keep the lowest:  lowest_score = a very large number
        #FOR each row i from 0 to n-1:
            #FOR each column j from i+1 to n-1:
                #score = (n-2) * D[i, j] - totals[i] - totals[j]
                #IF score < lowest_score:
                    #lowest_score = score
                    #best_i = i
                    #best_j = j

        #Step 3. Branch lengths for best_i and best_j
        #branch_i = (D[best_i, best_j] + (totals[best_i] - totals[best_j]) / (n-2)) / 2
        #branch_j = D[best_i, best_j] - branch_i
        #IF either branch is negative:
            #set it to 0 and add the difference to the other branch
        
        #Step 4. Distances from the new node to every other node k
        #new_distances = empty list
        #FOR each k that is not best_i or best_j:
            #new_distances.append((D[best_i, k] + D[best_j, k] - D[best_i, best_j]) / 2)

        #New_label for the joined pair:
        #new_label = "(" + current_labels[best_i] + ":" + branch_i + "," + current_labels[best_j] + ":" + branch_j + ")"

        #Step 5. Shrink the table
        #keep = every index except best_i and best_j
        #new_D = (n-1) x (n-1) grid of zeros
        #copy D's row/columns for the kept indices into the top left of new_D
        #put new_distances in the last row and last column of new_D (new node goes last)
        #current_labels = labels at the kept indices, then new_label added at the end
        #D = new_D

    #Step 6. Exactly 2 nodes left: connect them  (The final branch is split evenly between two sides so the tree can be drawn)
    #final_distance = D[0, 1] / 2
    #tree = "(" + current_labels[0] + ":" + final_distance + "," + current_labels[1] + ":" + final_distance + ");"  (4 decimals)

    #Return tree    

In [27]:
def plot_tree(newick_tree: str) -> None:
    """Plots a phylogenetic tree from a Newick string using Biopython and Matplotlib.

    The tree is rendered in a rectangular layout with smaller leaf labels and
    added margins to reduce overlap between labels and branches.

    Args:
        newick_tree (str): Tree in Newick format

    Returns:
        None: Displays the plotted tree
    """
    handle = StringIO(newick_tree)
    tree = Phylo.read(handle, "newick")

    fig, ax = plt.subplots(figsize=(8, 10))

    Phylo.draw(
        tree,
        axes=ax,
        do_show=False,
        label_func=lambda clade: clade.name if clade.is_terminal() else None,
    )
    for text in ax.texts:
        text.set_fontsize(6)              # shrink labels
    ax.margins(x=0.1, y=0.05)             # extra padding around tree

    plt.tight_layout()
    plt.show()


In [ ]:
if __name__ == "__main__":
    # Read HIV RT sequences
    sequences = read_fasta("lafayette_SARS_RT.fasta")
    
    # Build distance matrix using Smith-Waterman
    dist_matrix, seq_ids = build_distance_matrix(sequences)
    
    # Generate unrooted tree using Neighbor-Joining
    tree = neighbor_joining(dist_matrix, seq_ids)
    
    # Plot tree
    plot_tree(tree)